<a href="https://colab.research.google.com/github/KMKomer/Sentiment-analysis/blob/main/sentement_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
from transformers import (BertTokenizer, BertModel, get_linear_schedule_with_warmup)
import torch
from torch import nn
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW

In [3]:
df = pd.read_csv('drug_reviews.csv')
df.dropna()

le = LabelEncoder()

X_train, X_temp, Y_train, Y_temp = train_test_split(df['text'],df['sentiment'], random_state =42, test_size = 0.3)
X_val, X_test, Y_val, Y_test = train_test_split(X_temp,Y_temp, random_state =42, test_size = 0.5)

Y_train =le.fit_transform(Y_train)
Y_val = le.fit_transform(Y_val)
Y_test = le.fit_transform(Y_test)

device = torch.device('cpu')

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [4]:
class reviewdataset(Dataset):
    def __init__(self, reviews, labels, tokenizer, maxlen):
        self.reviews = reviews
        self.labels = labels
        self.tokenizer = tokenizer
        self.maxlen = maxlen

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        review = str(self.reviews[idx])
        label = self.labels[idx]


        tokens = self.tokenizer(review, add_special_tokens=True,return_token_type_ids=False,padding='max_length',max_length=self.maxlen, truncation =True,return_attention_mask=True, return_tensors ='pt')

        dic ={
            'input_ids': tokens['input_ids'].flatten(),
            'attention_mask':tokens['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

        return dic



In [5]:
train_dataset = reviewdataset(reviews=X_train.to_numpy(), labels = Y_train, tokenizer= tokenizer, maxlen=128)
val_dataset = reviewdataset(reviews=X_val.to_numpy(), labels=Y_val, tokenizer=tokenizer, maxlen=128)
test_dataset = reviewdataset(reviews=X_test.to_numpy(), labels=Y_test, tokenizer=tokenizer, maxlen=128)

train_loader = DataLoader(train_dataset, batch_size =16,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size = 16)
test_loader = DataLoader(test_dataset, batch_size = 16)
print(len(val_loader))


5


In [29]:
class classifier(nn.Module):
    def __init__(self, n_classes):
        super(classifier, self).__init__()
        self.bert = AutoModel.from_pretrained('distilbert-base-uncased', num_labels=3, output_hidden_states= False)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(256, n_classes)
        self.sigmoid = nn.Sigmoid()
        self.linear1 = nn.Linear(768, 512)
        self.linear2 = nn.Linear(512, 256)

    def forward(self, input_ids, attention_mask):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask,return_dict = False)
        last = output[0]
        output = last[:,0,:]
        out = self.drop(output)
        out = self.linear1(out)
        out = self.linear2(out)

        return self.out(out)


In [30]:
model = classifier(n_classes=3)
model = model.to(device)


In [37]:
optimizer = AdamW(model.parameters(), lr=3e-5, eps =1e-8)
loss_fn = nn.CrossEntropyLoss().to(device)
epouchs = 10
total_steps = len(train_loader) * epouchs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps = total_steps
)


def train(model,train_loader, optimizer,scheduler, device, loss_fn, n_examples):
    model.train()
    losses =[]
    corrected_preds = 0
    total_samples = 0
    all_preds = []
    all_labels = []

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=mask)
        _, preds= torch.max(outputs, dim=1)

        loss = loss_fn(outputs, labels)
        losses.append(loss.item())

        corrected_preds += torch.sum(preds == labels).item()
        total_samples += labels.size(0)

        model.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),max_norm = 1.0)

        optimizer.step()
        scheduler.step()

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    # metrics
    epoch_accuracy = corrected_preds / total_samples
    avg_epoch_loss = np.mean(losses)

    return epoch_accuracy, all_preds, all_labels, avg_epoch_loss


In [38]:
def evaluate(model, val_loader, device, loss_fn, n_examples):
    print('\nEvaluating...')
    model.eval()
    corrected_preds = 0
    losses = []
    total_samples = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
         for batch_idx, batch_data in enumerate(val_loader):
                input_ids = batch_data['input_ids'].to(device)
                mask = batch_data['attention_mask'].to(device)
                labels = batch_data['labels'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=mask)
                _,preds = torch.max(outputs, dim=1)
                loss = loss_fn(outputs, labels)
                losses.append(loss.item())

                corrected_preds += torch.sum(preds == labels).item()
                total_samples += labels.size(0)

                all_preds.extend(preds.cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

    # evaluation
    epoch_accuracy = corrected_preds / total_samples
    avg_epoch_loss = np.mean(losses)

    return epoch_accuracy, all_preds, all_labels, avg_epoch_loss


In [39]:
%%time

best_accuracy = 0

for epouch in range(epouchs):
    print(f'Epouch {epouch+1}/{epouchs}')
    print('='*70)
    train_acc, preds, labels ,mean_loss = train(model, train_loader, optimizer, scheduler, device, loss_fn, n_examples = len(X_train))
    print(f'Train accuracy : {train_acc:.4f}')
    print(f'Mean loss :{mean_loss:.4f}')
    

print('='*70)
val_accuracy, val_preds, val_labels, mean = evaluate(model, val_loader, device, loss_fn, n_examples = len(X_val))
print('Predictions vs Labels', val_preds, val_labels )
print(f'Validation accuracy : {val_accuracy:.4f}')
print(f'Mean loss : {mean:.4f}')



    
torch.save(model.state_dict(), 'sentimentanalyst.bin')

Epouch 1/10
Train accuracy : 0.8883
Mean loss :0.3469
Epouch 2/10
Train accuracy : 0.9742
Mean loss :0.1166
Epouch 3/10
Train accuracy : 0.9971
Mean loss :0.0299
Epouch 4/10
Train accuracy : 0.9943
Mean loss :0.0316
Epouch 5/10
Train accuracy : 0.9943
Mean loss :0.0225
Epouch 6/10
Train accuracy : 0.9971
Mean loss :0.0144
Epouch 7/10
Train accuracy : 0.9971
Mean loss :0.0148
Epouch 8/10
Train accuracy : 0.9971
Mean loss :0.0166
Epouch 9/10
Train accuracy : 1.0000
Mean loss :0.0028
Epouch 10/10
Train accuracy : 0.9971
Mean loss :0.0060

Evaluating...
Predictions vs Labels [1, 1, 1, 0, 1, 2, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 2, 0, 1, 2, 1, 2, 2, 1, 1, 1, 2, 2, 1, 2, 1, 1, 1, 0, 0, 2, 2, 2, 2, 1, 1, 2, 2, 2, 0, 1, 0, 2, 1, 2, 1, 1, 2, 1, 0, 0, 2, 2, 1, 1, 0, 2, 1, 1, 2, 2, 0, 2, 1, 2, 2, 1, 2, 2, 2] [1, 1, 1, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 2, 2, 1, 1, 2, 2, 1, 2, 1, 0, 0, 2, 0, 2, 2, 2, 2, 1, 1, 2, 2, 2, 0, 1, 0, 2, 2, 2, 1, 0, 1, 1, 0, 0, 2, 2, 1, 1, 0, 1, 1, 1, 2, 

In [42]:
print('===============Testing The Model==================')
test_accuracy, test_preds, test_labels, mean = evaluate(model, test_loader, device, loss_fn, n_examples = len(X_test))
print('Predictions vs Labels', test_preds, test_labels )
print(f'Testing accuracy : {test_accuracy:.4f}')
print(f'Mean loss : {mean:.4f}')

===============Testing The Model==================

Evaluating...
Predictions vs Labels [0, 2, 0, 1, 2, 1, 1, 0, 1, 0, 2, 2, 1, 2, 1, 0, 2, 1, 2, 1, 0, 1, 1, 2, 2, 1, 2, 1, 2, 2, 0, 2, 1, 2, 2, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 2, 1, 1, 1, 0, 1, 0, 2, 1, 2, 1, 1, 2, 0, 2, 0, 1, 1, 1, 2, 2, 1, 0, 2, 2, 0] [0, 2, 0, 0, 2, 1, 2, 0, 0, 0, 2, 1, 1, 2, 1, 0, 2, 0, 1, 1, 0, 1, 1, 0, 2, 1, 2, 1, 2, 1, 1, 2, 0, 2, 2, 1, 2, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 1, 0, 0, 1, 0, 0, 2, 1, 2, 1, 1, 2, 0, 2, 0, 1, 0, 1, 2, 2, 1, 0, 2, 2, 1]
Testing accuracy : 0.7600
Mean loss : 1.1532
